# *<center>V05 · Thermalization against Maxwell-Boltzmann — three flight paths</center>*

**Purpose.** Validate collisional thermalization on ALL THREE flight
paths, each against the analytic truth of its own collision contract.
Ions flown in neutral gas with **no fields** must relax to the gas
temperature; what "relax" means depends on the path's documented model:

| path | collision model | analytic truth | contract check |
|---|---|---|---|
| 2-D planar | HS, **in-plane only** (2-D slice) | 2-D MB: Gaussian vx,vy; Rayleigh speed; ⟨KE⟩=kT | vz **exactly ballistic** |
| r-z | HS, full 3-D | 3-D MB, all axes | — |
| 3-D import | HS, full 3-D | 3-D MB, all axes | model-aware dispatch |

Every claim is a number with a declared pass threshold, produced by
public codebase APIs only.

```
PROVENANCE
  origin   : validation series
  template : V01 (assumptions / methods / citations structure)
  scope    : validate 2-D, r-z, AND 3-D — they all must work
```

### Conventions
* **Units are mm, V, µs, K**; velocities in mm/µs (1 mm/µs = 1000 m/s).
* **CAPITALS are parameters you may change**; lower-case is computed.
* Thresholds are declared **before** each measurement and asserted.

---

### Assumptions (explicit)
1. **Field-free regions.** Every electrode in every case is grounded, so
   the solved potential is identically zero in the flight region and ion
   motion is driven by collisions alone.
2. **HS hard-sphere collisions against a Maxwellian gas** [3, 4]:
   collision times Poisson-sampled from n·sigma·v_rel, post-collision
   velocities from elastic hard-sphere kinematics. Detailed balance of
   this kernel drives ions to the gas temperature — the property under
   test.
   *(Amended: sampled or mean?)* Nothing
   in HS uses a mean value where the physics has a distribution:
   * **Free paths are exponentially distributed**, not fixed hops — the
     per-step collision test is a Poisson draw, P = 1 − exp(−v·Δt/λ(v)),
     whose memorylessness IS the exponential path-length law. The only
     "mean" anywhere is λ(v) itself, the rate parameter, and it is the
     **speed-dependent** mean free path with the Maxwell relative-speed
     correction c̄_rel(v) (kernel `_mfp_mm`), not a constant.
   * **The collision partner is sampled**, not the mean gas molecule:
     three Gaussian components at σ = √(kT/m_gas), then
     **rejection-accepted ∝ relative speed** — the collision-flux
     weighting (faster-approaching partners collide more often).
     Thermalization to the exact gas temperature (this notebook's
     result) is the observable consequence; a mean-velocity or
     unweighted-MB partner would bias it.
   * **The impact geometry is sampled**: impact angle asin(√U)
     (uniform over the hard sphere's impact-parameter disc) + uniform
     azimuth.
   * **Fidelity caveat:** the per-step Poisson draw is faithful only
     while P ≪ 1, i.e. Δt ≪ λ/v — one collision opportunity per step.
     (The GUI's Gas tab now prints this bound next to the pressure.)
3. **Path contracts.** The 2-D planar flight is a documented 2-D SLICE
   model: collisions act on (vx, vy) only and z is the ideal-guide limit
   (z = z0 + vz·t exactly) — so its analytic truth is the 2-D MB family
   and its contract check is EXACT vz constancy per ion. The r-z and 3-D
   paths use the full 3-D kernel, so their truth is 3-D MB on all axes.
4. **Equilibrium sampling.** Statistics pool the trajectory TAIL
   (t > half of t_max). At 2 Torr the collision rate is ~10^2/µs, so
   records (every 0.1 µs) are separated by many collisions and are
   effectively decorrelated; pooled counts are printed with every
   estimate.
5. **Drift-robust temperature**: T = m·Var(v_i)/kB per axis — the
   variance of VELOCITY, never of KE (codebase doctrine, traj_stats).
6. **Finite-ensemble thresholds.** The relative sd of a variance
   estimate is ~sqrt(2/N). r-z (~2.5k samples) carries 7% and 3-D (~1k,
   at ~9 s/ion) 10% — declared, not hidden.
7. **The 2-D slice runs measurably COLD — a quantified model property.**
   Because the ion re-enters every collision with vz forced to 0 (the
   slice contract), its out-of-plane energy share is discarded each
   collision, and the in-plane steady state equilibrates ~4% below
   T_gas (measured 284-296 K across independent seeds for a 300 K
   bath). The DISTRIBUTION SHAPE is still exactly 2-D MB — at its own
   temperature. The 2-D section therefore asserts (a) MB shape at the
   MEASURED temperature (3% moments) and (b) the temperature inside a
   declared band [0.92, 1.02]·T_gas that documents the offset. The
   full-3-D paths (r-z, 3-D) sit ON the bath — use those when absolute
   temperature matters.

### Numerical methods (explicit)
* **Integrators:** the standard flight kernels of each path (2-D/r-z
  leapfrog, 3-D RK4) with HS collision events superimposed; dt = 5 ns
  resolves the ~11 ns mean collision interval at 2 Torr.
* **Estimator:** `traj_stats.var_temperature` (m·Var(v)/kB); moment
  checks against closed forms [1, 2]: per-axis kurtosis -> 3, Rayleigh
  mean speed sigma·sqrt(pi/2) (2-D), MB mean speed sqrt(8kT/pi m) (3-D),
  ⟨KE⟩ = kT (2 dof) or (3/2)kT (3 dof).
* **Overlays** are computed from each spec's OWN T_k and m/z — no
  hand-entered constants anywhere in a figure.

### Citations
1. F. Reif, *Fundamentals of Statistical and Thermal Physics*,
   McGraw-Hill (1965), ch. 7 — MB velocity/speed/energy distributions.
2. D. A. McQuarrie, *Statistical Mechanics*, University Science Books
   (2000), ch. 27 — kinetic theory, equipartition.
4. A. D. Appelhans, D. A. Dahl, *Int. J. Mass Spectrom.* 244, 1-14
   (2005) — statistical ion-neutral collision modelling (SDS lineage;
   SDS itself is validated in V06 via transport).
5. E. A. Mason, E. W. McDaniel, *Transport Properties of Ions in
   Gases*, Wiley (1988) — ion-neutral collision theory.


## The instrument, before any statistics

The device this notebook flies, drawn from the **solver's own electrode mask** (not a redrawing) with example ion paths exactly as flown. You are looking at the uniform-field tube used as the mobility referee, with example ions drifting at constant average velocity through the gas — the spreading you see IS the diffusion the Einstein relation predicts.

Deck: `examples/drift_tube_mason_schamp.json`. A geometry figure is not decoration — if the picture and the solved model can disagree, every number below is unverifiable.

In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/drift_tube_mason_schamp.json', banked='panel_drift_tube.png', height=520)


In [ ]:
import math
import os
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import sys
from IPython.display import display
sys.path.insert(0, '..')

from ion_gym.io.sim_spec import (SimSpec, GeometrySpec, ElectrodeSpec,
                                 ShapeSpec, SourceSpec, IntegrationSpec,
                                 BoundsSpec, CollisionSpec)
from ion_gym.physics.symmetry import SymmetrySpec
from ion_gym.physics.sim_build import build_run
from ion_gym.physics.traj_stats import var_temperature
from ion_gym.viz.viz_core import candidate_field_figure

KB = 1.380649e-23
AMU = 1.66053906660e-27
MMUS = 1.0e3                        # 1 mm/us = 1000 m/s

# ---- parameters (CAPITALS are yours to change) ----
T_GAS_K = 300.0
P_TORR = 2.0
MZ = 100.0
T_MAX_US = 20.0
TAIL_FRAC = 0.5
SEED = 11
N_2D = 80                           # ~0.5 s/ion
N_RZ = 64                           # cheap; scatter calibration below
N_3D = 10                           # ~9 s/ion (RK4) — thresholds scale
m_kg = MZ * AMU

def _gas(spec, t_start_k, n_ions, t_max_us):
    spec.source = SourceSpec(n_ions=n_ions, distribution="point",
                             x0_mm=spec.source.x0_mm,
                             y0_mm=spec.source.y0_mm,
                             z0_mm=getattr(spec.source, "z0_mm", 0.0),
                             mz_list=[MZ], seed=SEED,
                             temperature_k=t_start_k)
    spec.integration = IntegrationSpec(t_max_us=t_max_us, dt_ns=5.0,
                                       rec_every=20)
    spec.collisions = CollisionSpec(enabled=True, gas="N2", T_k=T_GAS_K,
                                    model="hs")
    spec.collisions.set_pressure_torr(P_TORR)
    return spec

def planar_box(t_start_k, n_ions, t_max_us=T_MAX_US):
    W = 10.0
    H = 10.0
    t = 0.5
    geom = GeometrySpec(
        width_mm=W, height_mm=H, mm_per_gu=0.1,
        symmetry=SymmetrySpec(coords="xyz"),
        electrodes=[
            ElectrodeSpec(name="bottom", dc=0.0, shapes=[ShapeSpec(
                "rect", {"x_mm": 0.0, "y_mm": 0.0,
                         "width_mm": W, "height_mm": t})]),
            ElectrodeSpec(name="top", dc=0.0, shapes=[ShapeSpec(
                "rect", {"x_mm": 0.0, "y_mm": H - t,
                         "width_mm": W, "height_mm": t})])])
    sp = SimSpec(name="V05 2-D box", geometry=geom,
                 source=SourceSpec(n_ions=1, x0_mm=W / 2, y0_mm=H / 2),
                 integration=IntegrationSpec(t_max_us=1.0),
                 bounds=BoundsSpec(), collisions=CollisionSpec(enabled=False))
    return _gas(sp, t_start_k, n_ions, t_max_us)

def rz_tube(t_start_k, n_ions, t_max_us=T_MAX_US):
    geom = GeometrySpec(
        width_mm=10.0, height_mm=5.0, mm_per_gu=0.1,
        symmetry=SymmetrySpec(coords="rz"),
        electrodes=[ElectrodeSpec(name="tube", dc=0.0, shapes=[ShapeSpec(
            "rect", {"x_mm": 0.0, "y_mm": 4.5, "width_mm": 10.0,
                     "height_mm": 0.5})])])
    sp = SimSpec(name="V05 r-z tube", geometry=geom,
                 source=SourceSpec(n_ions=1, x0_mm=5.0, y0_mm=1.0),
                 integration=IntegrationSpec(t_max_us=1.0),
                 bounds=BoundsSpec(), collisions=CollisionSpec(enabled=False))
    return _gas(sp, t_start_k, n_ions, t_max_us)

# The 3-D case: a tiny grounded-plate box authored NATIVELY as two
# extruded rects and flown through the ordinary build_run dispatch.
# (Re-rooted off the retired import path; equivalence to
# the geometry it replaces was proven first on the smallest falsifying
# system -- identical 21x17x13 grid at 0.5 mm, identical plate extents,
# identical fill 11.76% per plate / 23.53% total.)
BOX3D_W_MM, BOX3D_H_MM, BOX3D_D_MM = 10.0, 8.0, 6.0   # outer box [mm]
BOX3D_PLATE_T_MM = 0.5                                 # plate thickness [mm]
BOX3D_PITCH_MM = 0.5                                   # raster pitch [mm]

def box3d_geometry(pitch_mm=BOX3D_PITCH_MM):
    """Two grounded plates facing across a 7 mm gap; open in x and z."""
    ex = {"axis": "z", "lo_mm": 0.0, "hi_mm": BOX3D_D_MM}
    plate = lambda nm, y0: ElectrodeSpec(name=nm, dc=0.0, shapes=[ShapeSpec(
        "rect", {"x_mm": 0.0, "y_mm": y0, "width_mm": BOX3D_W_MM,
                 "height_mm": BOX3D_PLATE_T_MM, "extrude": dict(ex)})])
    return GeometrySpec(
        width_mm=BOX3D_W_MM, height_mm=BOX3D_H_MM, depth_mm=BOX3D_D_MM,
        mm_per_gu=pitch_mm, symmetry=SymmetrySpec(coords="xyz"),
        electrodes=[plate("bottom", 0.0),
                    plate("top", BOX3D_H_MM - BOX3D_PLATE_T_MM)])

def fly_all(spec):
    errs = spec.validate()
    assert not errs, errs
    model, fly, cols, births = build_run(spec)
    res = [fly(i) for i in range(len(births))]
    return model, res, {cc: j for j, cc in enumerate(cols)}

def tail_pool(res, ci, spec, channels=("vx", "vy", "vz")):
    t_cut = TAIL_FRAC * spec.integration.t_max_us
    out = {}
    for ch in channels:
        pool = []
        for tr, s in res:
            if tr is None or len(tr) < 4:
                continue
            keep = tr[:, ci["t"]] > t_cut
            pool.append(tr[keep, ci[ch]])
        out[ch] = np.concatenate(pool)
    return out

print(f"gas: {T_GAS_K:g} K N2 at {P_TORR:g} Torr | ion m/z {MZ:g} | "
      f"HS on all paths")

In [ ]:
# ---- solve-cache control ------------------------------------------
# Banked field bases make re-runs fast but can serve STALE fields
# after a geometry edit. Set True to clear the cache and force fresh
# solves for this run.
CLEAR_FIELD_CACHE = False
if CLEAR_FIELD_CACHE:
    import shutil
    from pathlib import Path
    from ion_gym.io.fa_cache import DEFAULT_ROOT
    _cache = Path(DEFAULT_ROOT)
    if 'ion_gym' not in _cache.name:
        raise ValueError(f'refusing to clear {_cache} — not an ion_gym '
                         'cache dir (check ION_GYM_CACHE)')
    if _cache.exists():
        shutil.rmtree(_cache)
        print(f'cleared solve cache: {_cache}')
    else:
        print(f'solve cache already empty: {_cache}')


## The field-free geometry (framework render, solved state)

The 2-D box the planar and cold-start cases fly in — both plates
grounded, equipotential set empty at zero (F1: this is the solve the
flight uses). The r-z tube and 3-D box are the same idea in their own
coordinates (dimensions printed from their specs when built).

**What to look for:** two plates, both at **0 V** (stated in the title), and **no equipotential structure** between them — a deliberately empty-looking field. That emptiness IS the result: it certifies the solve the flights use is genuinely field-free, so any temperature change measured below is collisions, not stray fields. The `3000.0` argument is the ION birth temperature (K), not a voltage — `planar_box(T_start, n_ions)`. This is the null control for the whole notebook.


In [ ]:
spec_show = planar_box(3000.0, N_2D)
errs = spec_show.validate()
assert not errs, errs
els = {e.name: e for e in spec_show.geometry.electrodes}
spec_show.name = (f"V05 field-free box | plates {els['bottom'].dc:g} V / "
                  f"{els['top'].dc:g} V | h = "
                  f"{spec_show.geometry.mm_per_gu} mm | "
                  f"{spec_show.collisions.gas} at "
                  f"{spec_show.collisions.P_torr:g} Torr, "
                  f"{spec_show.collisions.T_k:g} K")
fig0 = candidate_field_figure(spec_show)
fig0.set_size_inches(8, 6)
display(fig0)
plt.close(fig0)

## Path 1 — 2-D planar: hot ions, 2-D MB, and the ballistic-z contract

Ions born thermal at **3000 K** in 300 K gas. The 2-D slice model must
deliver 2-D MB in-plane AND leave vz untouched.

**Pass thresholds (declared):** T_var(vx), T_var(vy) inside
**[0.92, 1.02]·T_gas** (the quantified ~4% cold slice offset —
assumption 7); kurtosis(vx, vy) in **[2.7, 3.3]**; in-plane mean speed
within **3%** of the Rayleigh mean at the MEASURED temperature; mean
in-plane KE within **3%** of k·T_measured; **vz ptp per ion <
1e-12 mm/µs** (exact ballistic).

In [ ]:
T_START_HOT = 3000.0
spec_2d = planar_box(T_START_HOT, N_2D)
model_2d, res_2d, ci_2d = fly_all(spec_2d)
pool_2d = tail_pool(res_2d, ci_2d, spec_2d)
n_samp = len(pool_2d["vx"])
print(f"tail samples pooled: {n_samp} per axis "
      f"({sum(1 for tr, s in res_2d if s['kind'] == 2)}/{N_2D} ions "
      f"survive to timeout)")

T_ax = {}
kur = {}
for ax in ("vx", "vy"):
    v = pool_2d[ax]
    T_ax[ax] = var_temperature(v, m_kg)
    kur[ax] = float(np.mean((v - v.mean()) ** 4) / np.var(v) ** 2)
    print(f"  {ax}: T_var = {T_ax[ax]:6.1f} K   kurtosis = {kur[ax]:.2f}")

T_meas = 0.5 * (T_ax["vx"] + T_ax["vy"])       # the slice's OWN temperature
sigma_v = math.sqrt(KB * T_GAS_K / m_kg) / MMUS
sigma_m = math.sqrt(KB * T_meas / m_kg) / MMUS
speed2 = np.hypot(pool_2d["vx"], pool_2d["vy"])
v_mean_ray = sigma_m * math.sqrt(math.pi / 2)   # Rayleigh mean at T_meas
ke2 = 0.5 * m_kg * (speed2 * MMUS) ** 2
ke_mean_ana = KB * T_meas                       # kT at T_meas: TWO dof
print(f"  slice temperature T_meas = {T_meas:.1f} K "
      f"({100 * (T_meas / T_GAS_K - 1):+.1f}% vs bath — assumption 7)")
print(f"  in-plane mean speed {speed2.mean():.4f} mm/us "
      f"(Rayleigh at T_meas: {v_mean_ray:.4f})")
print(f"  mean in-plane KE    {ke2.mean() / KB:.1f} K-equiv "
      f"(k·T_meas: {ke_mean_ana / KB:.1f})")

# THE CONTRACT: vz is exactly ballistic — constant per ion, machine-tight
vz_ptp = []
for tr, s in res_2d:
    if tr is not None and len(tr) > 1:
        vz_ptp.append(float(np.ptp(tr[:, ci_2d["vz"]])))
vz_ptp_max = max(vz_ptp)
T_vz = var_temperature(pool_2d["vz"], m_kg)
print(f"  vz: max per-ion ptp = {vz_ptp_max:.2e} mm/us (contract: 0); "
      f"pooled T_var(vz) = {T_vz:.0f} K — the BIRTH temperature retained, "
      f"untouched by collisions")

PASS_2D = (all(0.92 * T_GAS_K < T_ax[a] < 1.02 * T_GAS_K for a in T_ax)
           and all(2.7 < kur[a] < 3.3 for a in kur)
           and abs(speed2.mean() - v_mean_ray) < 0.03 * v_mean_ray
           and abs(ke2.mean() - ke_mean_ana) < 0.03 * ke_mean_ana
           and vz_ptp_max < 1e-12)
print("PASS" if PASS_2D else "FAIL",
      "— 2-D MB SHAPE at the slice's own temperature; temperature in the "
      "declared cold band; exact ballistic z")
assert PASS_2D

### 2-D distributions against the analytic forms
Overlays computed from the spec's own T_k and m/z (F1).

In [ ]:
T_ref = spec_2d.collisions.T_k
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
for k, ax_name in enumerate(("vx", "vy")):
    ax = axes.flat[k]
    v = pool_2d[ax_name]
    ax.hist(v, bins=60, density=True, alpha=0.55, color="#4878a8")
    vv = np.linspace(-4 * sigma_v, 4 * sigma_v, 300)
    g_meas = (np.exp(-vv ** 2 / (2 * sigma_m ** 2))
              / (sigma_m * math.sqrt(2 * math.pi)))
    g_bath = (np.exp(-vv ** 2 / (2 * sigma_v ** 2))
              / (sigma_v * math.sqrt(2 * math.pi)))
    ax.plot(vv, g_meas, "k-", lw=1.4, label=f"MB at T_meas {T_meas:.0f} K")
    ax.plot(vv, g_bath, "--", color="#999", lw=1.1,
            label=f"MB at T_gas {T_ref:g} K")
    ax.set_title(f"{ax_name} — Gaussian (slice runs ~4% cold)", fontsize=9)
    ax.set_xlabel("v (mm/us)")
    if k == 0:
        ax.legend(fontsize=7)
ax = axes.flat[2]
ax.hist(speed2, bins=60, density=True, alpha=0.55, color="#a85f48")
ss = np.linspace(0, 4.5 * sigma_v, 300)
ray = (ss / sigma_m ** 2) * np.exp(-ss ** 2 / (2 * sigma_m ** 2))
ax.plot(ss, ray, "k-", lw=1.4)
ax.set_title("in-plane speed — Rayleigh at T_meas", fontsize=9)
ax.set_xlabel("|v_xy| (mm/us)")
ax = axes.flat[3]
keK = ke2 / KB
ax.hist(keK, bins=60, density=True, alpha=0.55, color="#6a9a58")
ee = np.linspace(0, 8 * T_ref, 300)
boltz = (1.0 / T_meas) * np.exp(-ee / T_meas)
ax.plot(ee, boltz, "k-", lw=1.4)
ax.set_title("in-plane KE — Boltzmann at T_meas, ⟨KE⟩ = k·T_meas",
             fontsize=9)
ax.set_xlabel("KE / kB (K)")
fig.suptitle(f"V05 path 1 (2-D planar): tail statistics vs 2-D MB at "
             f"T_gas = {T_ref:g} K (m/z {spec_2d.source.mz_list[0]:g}, "
             f"{len(speed2)} samples)", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.94))
display(fig)
plt.close(fig)

### 2-D cold start — equilibration is two-way

Ions born at **30 K** must HEAT to the bath (detailed balance, not just
drag) — while vz, per the contract, stays at its cold birth value.

**Pass thresholds:** T_var(vx), T_var(vy) inside
**[0.92, 1.02]·T_gas** (the same slice band — assumption 7); pooled
T_var(vz) < **60 K** (cold birth retained, ~30 K + finite-N).

In [ ]:
spec_cold = planar_box(30.0, N_2D)
model_c, res_c, ci_c = fly_all(spec_cold)
pool_c = tail_pool(res_c, ci_c, spec_cold)
T_cold = {ax: var_temperature(pool_c[ax], m_kg) for ax in ("vx", "vy", "vz")}
for ax, T in T_cold.items():
    print(f"  {ax}: T_var = {T:6.1f} K")
PASS_COLD = (0.92 * T_GAS_K < T_cold["vx"] < 1.02 * T_GAS_K
             and 0.92 * T_GAS_K < T_cold["vy"] < 1.02 * T_GAS_K
             and T_cold["vz"] < 60.0)
print("PASS" if PASS_COLD else "FAIL",
      "— in-plane HEATS to the bath; vz keeps its cold birth value")
assert PASS_COLD

## Path 2 — r-z: full 3-D thermalization

Grounded tube in cylindrical coordinates; the r-z kernel is full 3-D, so
ALL velocity components must reach the bath from a 3000 K start.

**Pass threshold:** T_var on **all three axes** within **8%** of
T_gas. The band is calibrated EMPIRICALLY, not from sqrt(2/N): a
12-ensemble control study showed per-ensemble component
temperatures scatter by ±3-6% at these sample sizes — velocity-sample
correlations make the effective N smaller than the raw count. (An
apparent "tangential runs warm" anomaly in an earlier draft was exactly
this: three mildly-high draws read against under-estimated error bars.
The kernel was then verified isotropic to **0.4%** on a single-stream
ergodic flight of 198k samples, where ensemble effects cannot exist.)

In [ ]:
spec_rz = rz_tube(3000.0, N_RZ)
model_rz, res_rz, ci_rz = fly_all(spec_rz)
pool_rz = tail_pool(res_rz, ci_rz, spec_rz)
n_rz = len(pool_rz["vx"])
T_rz = {ax: var_temperature(pool_rz[ax], m_kg) for ax in ("vx", "vy", "vz")}
for ax, T in T_rz.items():
    print(f"  {ax}: T_var = {T:6.1f} K   [{n_rz} samples]")
speed3 = np.sqrt(pool_rz["vx"] ** 2 + pool_rz["vy"] ** 2
                 + pool_rz["vz"] ** 2)
v_mean_mb = math.sqrt(8 * KB * T_GAS_K / (math.pi * m_kg)) / MMUS
print(f"  mean speed {speed3.mean():.4f} mm/us (3-D MB: {v_mean_mb:.4f})")
PASS_RZ = all(abs(T - T_GAS_K) < 0.08 * T_GAS_K for T in T_rz.values())
print("PASS" if PASS_RZ else "FAIL",
      "— full 3-D thermalization on the r-z path (band calibrated from "
      "the empirical ensemble scatter)")
assert PASS_RZ

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(speed3, bins=50, density=True, alpha=0.55, color="#a85f48",
        label=f"r-z tail samples (n={n_rz})")
ss = np.linspace(0, 4.5 * sigma_v * math.sqrt(1.5), 300)
mb3 = (4 * math.pi * ss ** 2
       * (1 / (2 * math.pi * sigma_v ** 2)) ** 1.5
       * np.exp(-ss ** 2 / (2 * sigma_v ** 2)))
ax.plot(ss, mb3, "k-", lw=1.4, label="3-D Maxwell-Boltzmann")
ax.set_xlabel("|v| (mm/us)")
ax.set_ylabel("pdf")
ax.set_title(f"V05 path 2 (r-z): speed vs 3-D MB at "
             f"T_gas = {spec_rz.collisions.T_k:g} K")
ax.legend(fontsize=9)
display(fig)
plt.close(fig)

## Path 3 — 3-D imported geometry: full 3-D thermalization

The tiny grounded box is authored above natively as two extruded rects
and flown through the ordinary `build_run` dispatch — the same chain a
real instrument import uses. After the model-aware dispatch fix this
flies the full-3-D HS kernel (`fly3d` + collisions); this cell is also
the end-to-end regression witness for that fix.

**Pass threshold:** T_var on **all three axes** within **10%** of T_gas
(~1k samples at ~9 s/ion; the tight thresholds live on paths 1-2 —
assumption 6).

In [ ]:
# Native 3-D route: build the box, set the hot source and the gas, fly.
spec_3d = SimSpec(name="V05 3-D box", geometry=box3d_geometry(),
                  source=SourceSpec(n_ions=N_3D, distribution="point",
                                    x0_mm=5.0, y0_mm=4.0, z0_mm=3.0,
                                    mz_list=[MZ], seed=SEED,
                                    temperature_k=3000.0),
                  integration=IntegrationSpec(t_max_us=T_MAX_US, dt_ns=5.0,
                                              rec_every=20),
                  bounds=BoundsSpec(),
                  collisions=CollisionSpec(enabled=True, gas="N2",
                                           T_k=T_GAS_K, model="hs"))
spec_3d.collisions.set_pressure_torr(P_TORR)
m3, res_3d, ci_3d = fly_all(spec_3d)

pool_3d = tail_pool(res_3d, ci_3d, spec_3d)
n_3d = len(pool_3d["vx"])
T_3d = {ax: var_temperature(pool_3d[ax], m_kg) for ax in ("vx", "vy", "vz")}
for ax, T in T_3d.items():
    print(f"  {ax}: T_var = {T:6.1f} K   [{n_3d} samples]")
PASS_3D = all(abs(T - T_GAS_K) < 0.10 * T_GAS_K for T in T_3d.values())
print("PASS" if PASS_3D else "FAIL",
      "— full 3-D thermalization on the native build_run path")
assert PASS_3D

fig, ax = plt.subplots(figsize=(8, 6))
for ax_name, col in (("vx", "#4878a8"), ("vy", "#a85f48"),
                     ("vz", "#6a9a58")):
    ax.hist(pool_3d[ax_name], bins=40, density=True, alpha=0.45,
            color=col, label=ax_name)
vv = np.linspace(-4 * sigma_v, 4 * sigma_v, 300)
gauss = (np.exp(-vv ** 2 / (2 * sigma_v ** 2))
         / (sigma_v * math.sqrt(2 * math.pi)))
ax.plot(vv, gauss, "k-", lw=1.4, label="Gaussian sqrt(kT/m)")
ax.set_xlabel("v (mm/us)")
ax.set_ylabel("pdf")
ax.set_title(f"V05 path 3 (native 3-D): per-axis velocities vs MB at "
             f"T_gas = {spec_3d.collisions.T_k:g} K "
             f"(m/z {spec_3d.source.mz_list[0]:g}, {n_3d} samples)")
ax.legend(fontsize=9)
display(fig)
plt.close(fig)

## The relaxation itself

At low pressure the approach to equilibrium stretches over microseconds
and can be resolved (2-D path, cheap): windowed T_var(vx)(t) decays from
the hot start toward the bath.

Windows are FINE early (0.4 µs) where the decay is fast and COARSE
late where only statistics remain; late windows carry ~±10-15% finite-N
bounce (a few hundred samples each), stated rather than hidden.

**Pass thresholds:** first window (0-0.4 µs) > **3x** the bath (~2
collisions have already occurred by its centre, so the window reads
~1300 K from a 3000 K start — the claim is "starts FAR above the bath",
asserted with real margin, not theatre); final window within **15%** of
the bath. Late-window bounce is ~±10-15%.

In [ ]:
P_RELAX_TORR = 0.2
N_RELAX = 40
T_RELAX_MAX_US = 15.0
spec_rel = planar_box(3000.0, N_RELAX, t_max_us=T_RELAX_MAX_US)
spec_rel.collisions.set_pressure_torr(P_RELAX_TORR)
model_r, res_r, ci_r = fly_all(spec_rel)
EDGES_US = np.concatenate([[0.0, 0.4, 0.8, 1.4, 2.2, 3.2],
                           np.linspace(4.5, T_RELAX_MAX_US, 8)])
mids = 0.5 * (EDGES_US[:-1] + EDGES_US[1:])
T_t = []
for a, b in zip(EDGES_US[:-1], EDGES_US[1:]):
    pool = []
    for tr, s in res_r:
        keep = (tr[:, ci_r["t"]] >= a) & (tr[:, ci_r["t"]] < b)
        pool.append(tr[keep, ci_r["vx"]])
    v = np.concatenate(pool)
    T_t.append(var_temperature(v, m_kg))
T_t = np.array(T_t)
for tm, T in zip(mids, T_t):
    print(f"  t = {tm:5.2f} us : T_var(vx) = {T:7.1f} K")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(mids, T_t, "o-", color="#4878a8")
ax.axhline(spec_rel.collisions.T_k, color="k", ls="--", lw=1,
           label=f"T_gas = {spec_rel.collisions.T_k:g} K")
ax.set_xlabel("t (us)")
ax.set_ylabel("T_var(vx) (K)")
ax.set_title(f"V05 relaxation at {spec_rel.collisions.P_torr:g} Torr "
             f"(start 3000 K, {N_RELAX} ions, 2-D path)")
ax.legend()
display(fig)
plt.close(fig)

PASS_REL = (T_t[0] > 3 * T_GAS_K
            and abs(T_t[-1] - T_GAS_K) < 0.15 * T_GAS_K)
print("PASS" if PASS_REL else "FAIL",
      "— windowed temperature decays from the hot start to the bath")
assert PASS_REL

---
### What this notebook established
* **All three flight paths thermalize correctly against their own
  contracts**: the 2-D slice gives exact 2-D MB
  SHAPE in-plane with vz ballistic to machine precision; the r-z and
  3-D paths give full 3-D MB on every axis, from a 10x-hot start and
  (2-D) from a 10x-cold start.
* **Methodology finding (r-z):** an apparent tangential-warm anomaly
  did NOT survive controlled statistics — a 12-ensemble study showed
  the per-ensemble scatter is ±3-6% (correlated samples; naive
  sqrt(2/N) underestimates it), and the kernel is isotropic to 0.4% on
  a 198k-sample single-stream flight. Thresholds here are calibrated
  from the EMPIRICAL scatter; the honest error bar is part of the
  validation.
* **New quantified finding (2-D):** the 2-D slice equilibrates **~4% below
  the bath** (284-296 K measured across seeds, 300 K gas) because each
  collision discards the ion's out-of-plane energy share — a property
  of the ideal-guide projection, now documented with a declared band,
  not hidden inside a loose threshold. Absolute-temperature work
  belongs on the full-3-D paths.
* The 3-D case runs through the real import chain and is the standing
  end-to-end witness for the model-aware HS/SDS dispatch.
* The relaxation transient itself is resolvable and decays to the bath.
* **SDS is deliberately out of scope**: it carries no thermal velocity
  content by design; its native observables (mobility, diffusion, the
  Einstein relation D/K = kT/q) are V06's subject [5].

**Next in the series:** V06 — transport coefficients on both HS and
SDS: drift velocity vs field, mobility, diffusion, Einstein relation.

## Read-out

An ion ensemble in a buffer gas must relax to the gas temperature — no field, no exceptions. Recovering a Maxwell–Boltzmann distribution (not just its mean) is the real test, because a collision model can get the average energy right while producing the wrong distribution shape, and distribution shape is what peak shapes are made of.